# Python UDF Sandbox

This notebook demonstrates Databend's Python UDF (User Defined Function) capability:
- Creating Python functions in SQL
- Using UDFs for custom logic
- Agent orchestration patterns

In [ ]:
from databend_driver import connect

conn = connect("databend+local:///./local-state")

## 1. Create a simple Python UDF

UDFs let you write custom functions in Python that run inside Databend's sandbox.

In [ ]:
conn.execute("DROP FUNCTION IF EXISTS celsius_to_fahrenheit")
conn.execute("""
    CREATE FUNCTION celsius_to_fahrenheit(celsius FLOAT64)
    RETURNS FLOAT64
    LANGUAGE python
    HANDLER = 'convert'
    AS $$
    def convert(celsius):
        return celsius * 9.0 / 5.0 + 32.0
    $$
""")
print("Function created!")

In [ ]:
# Use the UDF in a query
result = conn.query("""
    SELECT temperature, celsius_to_fahrenheit(temperature) AS fahrenheit
    FROM (VALUES (0), (20), (37), (100)) AS t(temperature)
""")
print(result.fetchall())

## 2. String processing UDF

In [ ]:
conn.execute("DROP FUNCTION IF EXISTS slugify")
conn.execute("""
    CREATE FUNCTION slugify(input VARCHAR)
    RETURNS VARCHAR
    LANGUAGE python
    HANDLER = 'run'
    AS $$
    import re
    def run(input):
        slug = input.lower().strip()
        slug = re.sub(r'[^a-z0-9]+', '-', slug)
        slug = slug.strip('-')
        return slug
    $$
""")

result = conn.query("""
    SELECT slugify(title) AS slug
    FROM (VALUES 
        ('Hello World!'),
        ('Databend is Awesome'),
        ('Python UDFs 101')
    ) AS t(title)
""")
print(result.fetchall())

## 3. Agent orchestration pattern

UDFs can be used as building blocks for agent workflows. Each UDF handles one step, and SQL orchestrates the flow.

In [ ]:
# Create a simple "agent" function that classifies text
conn.execute("DROP FUNCTION IF EXISTS classify_text")
conn.execute("""
    CREATE FUNCTION classify_text(text VARCHAR)
    RETURNS VARCHAR
    LANGUAGE python
    HANDLER = 'run'
    AS $$
    def run(text):
        text_lower = text.lower()
        if any(w in text_lower for w in ['bug', 'error', 'fix']):
            return 'bug_report'
        elif any(w in text_lower for w in ['feature', 'add', 'new']):
            return 'feature_request'
        elif any(w in text_lower for w in ['help', 'how', 'question']):
            return 'question'
        return 'general'
    $$
""")

# Orchestrate: classify all incoming issues
result = conn.query("""
    SELECT issue_text, classify_text(issue_text) AS category
    FROM (VALUES
        ('Fix login bug on mobile'),
        ('Add dark mode support'),
        ('How to export data?'),
        ('Great product!')
    ) AS t(issue_text)
""")
print("Issue classification:")
for row in result.fetchall():
    print(f"  '{row[0]}' -> {row[1]}")